In [ ]:
import json, os
from typing import List, Dict, Any, Set

BASE = "arData"
PATHS = {
    "text": os.path.join(BASE, "text-legibility.json"),
    "pos":  os.path.join(BASE, "position.json"),
    "stab": os.path.join(BASE, "stability.json"),
}


OVERWRITE_ORIGINALS = True

def load_json(path: str) -> List[Dict[str, Any]]:
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    if not isinstance(data, list):
        raise ValueError(f"Erwartet: Liste von Records in {path}")
    return data

def get_device_id(rec: Dict[str, Any]):
    for k in ("deviceId", "device_id", "deviceid"):
        if k in rec and rec[k] not in (None, ""):
            return rec[k]
    for container in ("metrics", "formData", "meta"):
        sub = rec.get(container)
        if isinstance(sub, dict):
            for k in ("deviceId", "device_id", "deviceid"):
                v = sub.get(k)
                if v not in (None, ""):
                    return v
    return None

def ids_in(records: List[Dict[str, Any]]) -> Set[Any]:
    out = set()
    for r in records:
        did = get_device_id(r)
        if did is not None:
            out.add(did)
    return out

def filter_by_ids(records: List[Dict[str, Any]], keep_ids: Set[Any]) -> List[Dict[str, Any]]:
    return [r for r in records if get_device_id(r) in keep_ids]

# Laden
text_raw = load_json(PATHS["text"])
pos_raw  = load_json(PATHS["pos"])
stab_raw = load_json(PATHS["stab"])

# Schnittmenge bilden
ids_text = ids_in(text_raw)
ids_pos  = ids_in(pos_raw)
ids_stab = ids_in(stab_raw)
common_ids = ids_text & ids_pos & ids_stab

print(f"Geräte-IDs: text={len(ids_text)} | position={len(ids_pos)} | stability={len(ids_stab)}")
print(f"Schnittmenge (in allen 3 vorhanden): {len(common_ids)}")

missing_in_text = (ids_pos & ids_stab) - ids_text
missing_in_pos  = (ids_text & ids_stab) - ids_pos
missing_in_stab = (ids_text & ids_pos)  - ids_stab
if missing_in_text:
    print("IDs fehlen in text-legibility.json:", sorted(missing_in_text))
if missing_in_pos:
    print("IDs fehlen in position.json:", sorted(missing_in_pos))
if missing_in_stab:
    print("IDs fehlen in stability.json:", sorted(missing_in_stab))

# Filtern
text_f = filter_by_ids(text_raw, common_ids)
pos_f  = filter_by_ids(pos_raw,  common_ids)
stab_f = filter_by_ids(stab_raw, common_ids)

print(f"Nach Filter: text={len(text_f)} | position={len(pos_f)} | stability={len(stab_f)}")

# Speichern
if OVERWRITE_ORIGINALS:
    # Destruktiv: Originale ersetzen
    with open(PATHS["text"], "w", encoding="utf-8") as f: json.dump(text_f, f, ensure_ascii=False, indent=2)
    with open(PATHS["pos"],  "w", encoding="utf-8") as f: json.dump(pos_f,  f, ensure_ascii=False, indent=2)
    with open(PATHS["stab"], "w", encoding="utf-8") as f: json.dump(stab_f, f, ensure_ascii=False, indent=2)
    print("Originaldateien wurden überschrieben.")
else:
    # Nicht-destruktiv: Kopien schreiben und zusätzlich 'data' für die nächste Zelle bereitstellen
    out_dir = os.path.join(BASE, "filtered")
    os.makedirs(out_dir, exist_ok=True)
    with open(os.path.join(out_dir, "text-legibility.json"), "w", encoding="utf-8") as f: json.dump(text_f, ensure_ascii=False, indent=2)
    with open(os.path.join(out_dir, "position.json"),        "w", encoding="utf-8") as f: json.dump(pos_f,  ensure_ascii=False, indent=2)
    with open(os.path.join(out_dir, "stability.json"),       "w", encoding="utf-8") as f: json.dump(stab_f, ensure_ascii=False, indent=2)
    print(f"Gefilterte Kopien unter: {out_dir}")

data = text_f


In [ ]:

import json, numpy as np, pandas as pd

PATH = "arData/text-legibility.json"

with open(PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

def find_results_in_interactions(rec):
    metrics = rec.get("metrics", {})
    interactions = metrics.get("interactions", rec.get("interactions", [])) or []
    candidates = []
    for ev in interactions:
        if not isinstance(ev, dict):
            continue
        val = ev.get("value")
        if isinstance(val, dict):
            res = val.get("results")
            if isinstance(res, dict):
                mr = res.get("minReadableSize")
                xr = res.get("maxReadableSize")
                cr = res.get("comfortableReadableSize")
                if any(isinstance(v,(int,float)) for v in [mr,xr,cr]):
                    candidates.append({"min": mr, "max": xr, "comfort": cr})
    return candidates[-1] if candidates else None

rows = []
for i, rec in enumerate(data):
    res = find_results_in_interactions(rec)
    if res:
        rows.append({
            "X": i,
            "minReadableSize": res.get("min"),
            "maxReadableSize": res.get("max"),
            "comfortableReadableSize": res.get("comfort"),
        })
df = pd.DataFrame(rows)
df.head()

In [ ]:

metrics_cols = ["minReadableSize", "maxReadableSize", "comfortableReadableSize"]
summary = {}
for col in metrics_cols:
    s = df[col].dropna()
    summary[f"{col} – mean"] = s.mean() if not s.empty else None
    summary[f"{col} – median"] = s.median() if not s.empty else None
pd.DataFrame([summary])

In [ ]:

import matplotlib.pyplot as plt

for col in ["minReadableSize","maxReadableSize","comfortableReadableSize"]:
    s = df[col].dropna()
    if s.empty: 
        continue
    plt.figure()
    plt.hist(s, bins=10)
    plt.xlabel(col)
    plt.ylabel("Häufigkeit")
    plt.title(f"Histogramm – {col}")
    plt.show()

In [ ]:

import matplotlib.pyplot as plt

for col in ["minReadableSize","maxReadableSize","comfortableReadableSize"]:
    s = df[col].dropna()
    if s.empty:
        continue
    plt.figure()
    plt.boxplot(s, vert=True)
    plt.ylabel(col)
    plt.title(f"Boxplot – {col}")
    plt.show()